# Data Quality Audit — 01 · Tourism Reviews (Zenodo)

**Source:** `data/raw/tourism_reviews/golden_data_set_processed_google_reviews.xlsx` — Arabic, CC0

**Purpose in the concierge:** Review evidence + aspect sentiment for personalization.

Standardized audit covering:

```
Dataset
├── Shape
├── Columns & data types
├── Missing values
├── Duplicates
├── Invalid values
├── Outliers
├── Inconsistent categories
├── Geographic validity
├── Date/time validity
├── Data-source/license
└── Known limitations
```

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# Resolve repo root whether run from the audit folder or the repo root.
p = Path.cwd()
while p != p.parent and not (p / "data" / "raw").exists():
    p = p.parent
ROOT = p
print("repo root:", ROOT)

# Saudi Arabia bounding box (approx) for geographic validity checks.
SA_LAT = (16.0, 32.5)
SA_LON = (34.5, 56.0)

def iqr_outliers(series):
    """Return (count, lower, upper) of IQR outliers in a numeric series."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return 0, np.nan, np.nan
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum()), round(lo, 2), round(hi, 2)

def missing_report(df):
    m = pd.DataFrame({"missing": df.isna().sum(),
                      "missing_%": (df.isna().mean() * 100).round(1)})
    return m[m["missing"] > 0].sort_values("missing", ascending=False)

def dtype_report(df):
    return pd.DataFrame({
        "dtype": [str(t) for t in df.dtypes],
        "non_null": df.notna().sum().values,
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    }, index=df.columns)


repo root: /home/user/saudi-Digital-Concierge


In [2]:
df = pd.read_excel(ROOT / "data/raw/tourism_reviews/golden_data_set_processed_google_reviews.xlsx")
GOLD = [c for c in df.columns if "golden" in c]
print("Loaded reviews:", df.shape)
df.head(3)

Loaded reviews: (3543, 10)


,اسم المكان,processed_reviews,المدينة,الفئة,السعر_golden,النظافة_golden,المرافق_golden,الخدمة و الموظفين_golden,البيئة_golden,التجربة بشكل عام_golden
0,منتزه الخلب,منتزه وحديقة البرج مكان جميل بعد المندق مررنا ...,الباحة,منتزه,1.0,-1.0,1.0,NaN,1,1.0
1,منتزه الخلب,مكان جميل يهمني وجود العاب للاطفالعندهم مجموعة...,الباحة,منتزه,1.0,NaN,1.0,NaN,1,1.0
2,منتزه الخلب,موقع جميلمطل على منظر حلوولكن يحتاج اهتمام اكث...,الباحة,منتزه,1.0,NaN,1.0,NaN,1,1.0


## Shape

In [3]:
print("Rows:", len(df), "| Columns:", df.shape[1])

Rows: 3543 | Columns: 10


## Columns & data types
Columns are Arabic; six `*_golden` aspect-sentiment scores.

In [4]:
dtype_report(df)

,dtype,non_null,n_unique
اسم المكان,str,3543,42
processed_reviews,str,3543,3531
المدينة,str,3542,17
الفئة,str,3543,4
السعر_golden,float64,1119,3
النظافة_golden,float64,482,3
المرافق_golden,float64,1287,3
الخدمة و الموظفين_golden,float64,663,3
البيئة_golden,object,3167,4
التجربة بشكل عام_golden,float64,2244,3


## Missing values
Aspect scores are only populated when that aspect is mentioned, so high missingness is expected (not corruption).

In [5]:
missing_report(df)

,missing,missing_%
النظافة_golden,3061,86.4
الخدمة و الموظفين_golden,2880,81.3
السعر_golden,2424,68.4
المرافق_golden,2256,63.7
التجربة بشكل عام_golden,1299,36.7
البيئة_golden,376,10.6
المدينة,1,0.0


## Duplicates

In [6]:
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate on (place, review):", df.duplicated(subset=["اسم المكان","processed_reviews"]).sum())

Exact duplicate rows: 5
Duplicate on (place, review): 6


## Invalid values
Golden aspect scores must be in {-1, 0, 1}. Any other value (or a non-numeric type) is invalid.

In [7]:
for col in GOLD:
    vals = pd.to_numeric(df[col], errors="coerce")
    bad_type = df[col].notna().sum() - vals.notna().sum()  # non-numeric present
    out_of_set = (~vals.dropna().isin([-1, 0, 1])).sum()
    print(f"{col:28s} dtype={str(df[col].dtype):8s} non_numeric={bad_type} out_of_[-1,0,1]={out_of_set}")

السعر_golden                 dtype=float64  non_numeric=0 out_of_[-1,0,1]=0
النظافة_golden               dtype=float64  non_numeric=0 out_of_[-1,0,1]=0
المرافق_golden               dtype=float64  non_numeric=0 out_of_[-1,0,1]=0
الخدمة و الموظفين_golden     dtype=float64  non_numeric=0 out_of_[-1,0,1]=0
البيئة_golden                dtype=object   non_numeric=1 out_of_[-1,0,1]=0
التجربة بشكل عام_golden      dtype=float64  non_numeric=0 out_of_[-1,0,1]=0


## Outliers
Scores are categorical (-1/0/1), so classic numeric outliers do not apply. We instead show the value distribution per aspect.

In [8]:
for col in GOLD:
    print(col, "->", pd.to_numeric(df[col], errors="coerce").value_counts(dropna=True).to_dict())

السعر_golden -> {-1.0: 916, 1.0: 133, 0.0: 70}
النظافة_golden -> {1.0: 314, -1.0: 162, 0.0: 6}
المرافق_golden -> {1.0: 763, -1.0: 413, 0.0: 111}
الخدمة و الموظفين_golden -> {1.0: 431, -1.0: 219, 0.0: 13}
البيئة_golden -> {1.0: 2699, -1.0: 345, 0.0: 122}
التجربة بشكل عام_golden -> {1.0: 1465, -1.0: 548, 0.0: 231}


## Inconsistent categories
City (`المدينة`) has spelling/spacing variants; category (`الفئة`) is small and clean.

In [9]:
print("City — %d unique:" % df["المدينة"].nunique())
print(df["المدينة"].value_counts(dropna=False).to_string())
print("\nCategory — %d unique:" % df["الفئة"].nunique())
print(df["الفئة"].value_counts(dropna=False).to_string())

City — 17 unique:
المدينة
ابها                                764
الباحة                              588
الطائف                              470
الاحساء                             303
المدينة المنورة                     296
ينبع                                214
الدمام                              169
الرياض                              150
المدينه المنورة                     131
الخبر                               120
مدينة الملك عبد الله الاقتصادية      99
جازان                                88
العلا                                64
أبها                                 30
حائل                                 30
جده                                  14
تبوك                                 12
NaN                                   1

Category — 4 unique:
الفئة
حديقة                  1275
مكان تراثي وتاريخي     1084
منتزه                   773
متحف                    411


## Geographic validity
No latitude/longitude columns — geographic validity is **N/A** (city text only, to be mapped to a canonical key in cleaning).

In [10]:
print("Has coordinates:", any("lat" in c.lower() or "lon" in c.lower() for c in df.columns))

Has coordinates: False


## Date/time validity
No date/time columns — **N/A**.

In [11]:
print("Date-like columns:", [c for c in df.columns if any(k in c.lower() for k in ["date","time","year","تاريخ"])])

Date-like columns: []


## Data-source / license
- **Source:** Zenodo — *Saudi Tourism Reviews*.
- **License:** CC0 (public domain).
- **Currency:** static snapshot (not live).

## Known limitations
- **High missingness** in aspect scores (each aspect only scored when mentioned).
- **`البيئة_golden` is object dtype** → contains a non-numeric value to coerce/clean.
- **Inconsistent city spellings** (`جده`/`جدة`, `المدينه`/`المدينة المنورة `, trailing spaces).
- **5 duplicate rows** to drop.
- Arabic text needs Unicode normalization (NFKC, diacritics) before embedding.
- No coordinates or dates; city is free text.